# Reproducción del análisis 4/5 en Obrajes (Soracachi, Oruro)
Datos: NASA POWER diario + ONI de NOAA. Ejecutar: Entorno de ejecución → Ejecutar todas. Copiar la salida de las celdas 4, 5 y 8 a P6.


In [ ]:
# =============================================================
# PRUEBA DE DESCARTE - Señal El Niño y heladas en Oruro
# Correr en Google Colab: pegar todo en UNA celda y presionar ▶
# =============================================================
import requests
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ---------- 1. CONFIGURACIÓN (cambiar cuando sepamos el municipio) ----------
LAT = -17.83   # Obrajes, Soracachi (parcela del productor)
LON = -67.00
NOMBRE = "Obrajes"
UMBRAL_HELADA = -2.5   # °C: papa común sin aclimatar (Chen y Li 1980)

In [ ]:
# ---------- 2. BAJAR DATOS DE NASA POWER (diarios desde 1981) ----------
url = ("https://power.larc.nasa.gov/api/temporal/daily/point"
       "?parameters=T2M_MIN,PRECTOTCORR&community=AG"
       f"&longitude={LON}&latitude={LAT}"
       "&start=19810101&end=20261110&format=JSON")
print("Descargando NASA POWER... (puede tardar 30-60 s)")
datos = requests.get(url, timeout=180).json()["properties"]["parameter"]

df = pd.DataFrame({
    "tmin": datos["T2M_MIN"],
    "lluvia": datos["PRECTOTCORR"],
})
df.index = pd.to_datetime(df.index, format="%Y%m%d")
df = df.replace(-999, float("nan"))          # -999 = dato faltante
df.to_csv(f"power_{NOMBRE}.csv")             # guardamos copia (Plan B offline)
print("Días descargados:", len(df), "| último día:", df.index.max().date())

In [ ]:
# ---------- 3. BAJAR ÍNDICE ONI (NOAA) para saber qué años fueron Niño ----------
oni = pd.read_csv("https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt",
                  sep=r"\s+")
oni = oni[oni["SEAS"] == "NDJ"]               # trimestre nov-dic-ene
oni = oni.set_index("YR")["ANOM"]

def fase(valor):
    if valor >= 0.5:
        return "Niño"
    if valor <= -0.5:
        return "Niña"
    return "Normal"

In [ ]:
# ---------- 4. LLUVIA POR CAMPAÑA (noviembre a marzo) ----------
filas = []
for anio in range(1981, 2026):
    inicio = f"{anio}-11-01"
    fin = f"{anio + 1}-03-31"
    tramo = df.loc[inicio:fin]
    if len(tramo) < 140 or anio not in oni.index:
        continue
    filas.append({
        "campaña": f"{anio}-{str(anio + 1)[2:]}",
        "fase": fase(oni[anio]),
        "lluvia_mm": round(tramo["lluvia"].sum()),
        "dias_helada": int((tramo["tmin"] <= UMBRAL_HELADA).sum()),
    })
camp = pd.DataFrame(filas)
camp.to_csv(f"campañas_{NOMBRE}.csv", index=False)

print("\n=== PROMEDIO POR FASE (campaña nov-mar) ===")
print(camp.groupby("fase")[["lluvia_mm", "dias_helada"]].mean().round(1))
print("\nCantidad de campañas por fase:")
print(camp["fase"].value_counts())

In [ ]:
# ---------- 5. HELADAS EN FEBRERO ----------
feb = df[df.index.month == 2]
anios_con_helada = feb[feb["tmin"] <= UMBRAL_HELADA].index.year.unique()
total_anios = feb.index.year.nunique()
print(f"\nFebreros con al menos una helada <= {UMBRAL_HELADA} °C:",
      f"{len(anios_con_helada)} de {total_anios}",
      f"({100 * len(anios_con_helada) / total_anios:.0f} %)")

In [ ]:
# ---------- 6. ¿APARECE LA HELADA DEL 3 DE FEBRERO DE 2026? ----------
print("\nTemperatura mínima 1-5 de febrero de 2026:")
print(df.loc["2026-02-01":"2026-02-05", "tmin"])

In [ ]:
# ---------- 7. GRÁFICO ----------
colores = {"Niño": "tab:red", "Niña": "tab:blue", "Normal": "tab:gray"}
plt.figure(figsize=(12, 4))
plt.bar(camp["campaña"], camp["lluvia_mm"],
        color=[colores[f] for f in camp["fase"]])
plt.xticks(rotation=90)
plt.ylabel("Lluvia nov-mar (mm)")
plt.title(f"{NOMBRE}: lluvia por campaña (rojo = Niño, azul = Niña) - NASA POWER")
plt.tight_layout()
plt.savefig(f"lluvia_campañas_{NOMBRE}.png", dpi=150)
plt.show()

In [ ]:
# ---------- 8. NIÑO FUERTE (ONI >= 1.5) Y MÉTODO ----------
camp["oni"] = [oni[int(c[:4])] for c in camp["campaña"]]
promedio_normal = camp[camp["fase"] == "Normal"]["lluvia_mm"].mean()
camp["vs_normal_%"] = ((camp["lluvia_mm"] / promedio_normal - 1) * 100).round()
fuertes = camp[camp["oni"] >= 1.5]
print("Normal de referencia (promedio campañas Normales):", round(promedio_normal), "mm, n =", (camp["fase"] == "Normal").sum())
print(fuertes[["campaña", "oni", "lluvia_mm", "vs_normal_%"]])
secas = (fuertes["lluvia_mm"] < promedio_normal).sum()
print(f"\nNiño fuerte con lluvia bajo lo normal: {secas} de {len(fuertes)}")
camp.to_csv(f"campañas_{NOMBRE}.csv", index=False)
